# Economic Load Dispatch with Prohibited Operating Zones

**Solve power dispatch with quantum computing using the Divi SDK**

---

This notebook dispatches power across **3 generators** to meet a **195 MW demand** at minimum fuel cost, while avoiding Prohibited Operating Zones (mechanical vibration bands).

The pipeline:
1. **Define generators** — cost curves, power ranges, and forbidden zones
2. **Build a QUBO** — encode the problem as 12 binary variables
3. **Solve with PCE-VQE** — a quantum solver that compresses 12 variables into 5 qubits
4. **Repair** — fix near-feasible quantum candidates into fully valid dispatches
5. **Compare** — check against the classical brute-force optimum

In [ ]:
import dimod
import numpy as np
import pennylane as qml

from divi.backends import ParallelSimulator
from divi.qprog import PCE, GenericLayerAnsatz
from divi.qprog.optimizers import PymooMethod, PymooOptimizer
from divi.typing import qubo_to_matrix

## Step 1 — Define the generators

In [ ]:
def define_generators():
    """Define the 3-generator microgrid.

    Each generator has:
      - a, b, c:          fuel cost curve  Cost = a + b·P + c·P²
      - P_min, P_max:     operating range in MW
      - poz_low, poz_high: prohibited operating zone (vibration band)
    """
    return [
        {"name": "Gen 1", "a": 20, "b": 2.0, "c": 0.010,
         "P_min": 40, "P_max": 115, "poz_low": 60, "poz_high": 75},
        {"name": "Gen 2", "a": 15, "b": 1.5, "c": 0.020,
         "P_min": 20, "P_max": 95,  "poz_low": 40, "poz_high": 55},
        {"name": "Gen 3", "a": 25, "b": 1.8, "c": 0.015,
         "P_min": 30, "P_max": 105, "poz_low": 50, "poz_high": 65},
    ]


generators = define_generators()
DEMAND = 195  # MW

for gen in generators:
    print(f"  {gen['name']}:  {gen['P_min']}–{gen['P_max']} MW,  "
          f"POZ: {gen['poz_low']}–{gen['poz_high']} MW")

## Step 2 — Build the QUBO

We discretise each generator's output into 16 levels (4 binary variables, 5 MW apart) and encode three objectives into a single energy function:
- **Fuel cost** — what we want to minimise
- **Demand penalty** — forces total generation = demand
- **POZ penalty** — forbids operation in vibration bands

In [ ]:
STEP_MW = 5
N_QUBITS_PER_GEN = 4
BIT_WEIGHTS = [2**b for b in range(N_QUBITS_PER_GEN)]  # [1, 2, 4, 8]


def fuel_cost(gen, power):
    """Compute fuel cost ($) for a generator at a given power (MW)."""
    return gen["a"] + gen["b"] * power + gen["c"] * power ** 2


def _qubit_name(gen_idx, bit_idx):
    return f"q_{gen_idx}_{bit_idx}"


def decode_power(gen_idx, generators, bit_values):
    """Convert binary qubit values back to MW for one generator."""
    integer_val = sum(
        BIT_WEIGHTS[b] * bit_values[_qubit_name(gen_idx, b)]
        for b in range(N_QUBITS_PER_GEN)
    )
    return generators[gen_idx]["P_min"] + STEP_MW * integer_val


def build_qubo(generators, demand, penalty_lambda=2000, poz_mu=5000):
    """Build the Binary Quadratic Model (BQM) for the ELD problem."""
    bqm = dimod.BinaryQuadraticModel(vartype="BINARY")
    var_names = []
    for g in range(len(generators)):
        for b in range(N_QUBITS_PER_GEN):
            var_names.append(_qubit_name(g, b))

    # 1. Fuel cost objective
    cost_offset = 0.0
    for g, gen in enumerate(generators):
        a, b_coeff, c = gen["a"], gen["b"], gen["c"]
        p_min = gen["P_min"]
        cost_offset += a + b_coeff * p_min + c * p_min ** 2
        for k in range(N_QUBITS_PER_GEN):
            w_k = STEP_MW * BIT_WEIGHTS[k]
            var_k = _qubit_name(g, k)
            bqm.add_linear(var_k, b_coeff * w_k + c * (2 * p_min * w_k + w_k ** 2))
            for l in range(k + 1, N_QUBITS_PER_GEN):
                w_l = STEP_MW * BIT_WEIGHTS[l]
                bqm.add_quadratic(var_k, _qubit_name(g, l), c * 2 * w_k * w_l)
    bqm.offset += cost_offset

    # 2. Demand constraint
    d_const = sum(gen["P_min"] for gen in generators) - demand
    bqm.offset += penalty_lambda * d_const ** 2
    d_terms = [(_qubit_name(g, k), STEP_MW * BIT_WEIGHTS[k])
               for g in range(len(generators)) for k in range(N_QUBITS_PER_GEN)]
    for var_i, w_i in d_terms:
        bqm.add_linear(var_i, penalty_lambda * (2 * d_const * w_i + w_i ** 2))
    for i in range(len(d_terms)):
        vi, wi = d_terms[i]
        for j in range(i + 1, len(d_terms)):
            vj, wj = d_terms[j]
            bqm.add_quadratic(vi, vj, penalty_lambda * 2 * wi * wj)

    # 3. POZ penalty
    for g in range(len(generators)):
        bqm.add_linear(_qubit_name(g, 2), poz_mu)
        bqm.add_quadratic(_qubit_name(g, 3), _qubit_name(g, 2), -poz_mu)

    return bqm, var_names


bqm, var_names = build_qubo(generators, demand=DEMAND)
print(f"Built QUBO: {len(var_names)} variables, {len(bqm.quadratic)} interactions")

## Step 3 — Classical brute force (for comparison)

In [ ]:
def classical_brute_force(generators, demand):
    """Enumerate all 4,096 configurations and return the cheapest valid one."""
    best = None
    for i1 in range(16):
        p1 = generators[0]["P_min"] + STEP_MW * i1
        poz1 = generators[0]["poz_low"] <= p1 <= generators[0]["poz_high"]
        for i2 in range(16):
            p2 = generators[1]["P_min"] + STEP_MW * i2
            poz2 = generators[1]["poz_low"] <= p2 <= generators[1]["poz_high"]
            for i3 in range(16):
                p3 = generators[2]["P_min"] + STEP_MW * i3
                poz3 = generators[2]["poz_low"] <= p3 <= generators[2]["poz_high"]
                if p1 + p2 + p3 != demand or poz1 or poz2 or poz3:
                    continue
                cost = sum(fuel_cost(generators[g], p)
                           for g, p in enumerate([p1, p2, p3]))
                if best is None or cost < best[3]:
                    best = (p1, p2, p3, cost)
    return best


classical_best = classical_brute_force(generators, DEMAND)
print(f"Classical optimum: P1={classical_best[0]}, P2={classical_best[1]}, "
      f"P3={classical_best[2]} MW  → Cost = {classical_best[3]:.1f} $")

## Step 4 — Solve with quantum computing (PCE-VQE)

PCE compresses 12 QUBO variables into just **5 qubits** using polynomial encoding.

In [ ]:
def solve_with_pce(bqm, n_layers=3, max_iterations=20, alpha=3.0,
                   population_size=30, shots=10000):
    """Run the PCE-VQE quantum solver on the BQM."""
    qubo_mat = qubo_to_matrix(bqm)
    ansatz = GenericLayerAnsatz(
        gate_sequence=[qml.RY, qml.RZ],
        entangler=qml.CNOT,
        entangling_layout="all-to-all",
    )
    pce_solver = PCE(
        qubo_matrix=qubo_mat,
        ansatz=ansatz,
        n_layers=n_layers,
        encoding_type="poly",
        optimizer=PymooOptimizer(method=PymooMethod.DE,
                                 population_size=population_size),
        max_iterations=max_iterations,
        alpha=alpha,
        backend=ParallelSimulator(shots=shots),
    )
    print(f"PCE qubits: {pce_solver.n_qubits}  (poly encoding of {len(bqm.variables)} variables)")
    pce_solver.run()
    return pce_solver


print("🚀 Running quantum solver...")
pce_solver = solve_with_pce(bqm)

## Step 5 — Repair quantum solutions

Quantum solvers produce near-feasible candidates. A lightweight classical **repair** step:
1. Snaps POZ-violating generators to the nearest allowed level
2. Greedily adjusts generators to match demand exactly

In [ ]:
def repair_solution(powers, generators, demand):
    """Fix a near-feasible quantum solution so it meets all constraints."""
    ps = list(powers)

    # Stage 1: fix POZ violations
    for g, gen in enumerate(generators):
        if gen["poz_low"] <= ps[g] <= gen["poz_high"]:
            allowed = [
                gen["P_min"] + STEP_MW * idx
                for idx in range(2 ** N_QUBITS_PER_GEN)
                if not (gen["poz_low"] <= gen["P_min"] + STEP_MW * idx <= gen["poz_high"])
            ]
            ps[g] = min(allowed, key=lambda lv: abs(lv - ps[g]))

    # Stage 2: fix demand mismatch
    for _ in range(50):
        gap = demand - sum(ps)
        if gap == 0:
            break
        step = STEP_MW if gap > 0 else -STEP_MW
        best_g, best_cost = None, float("inf")
        for g, gen in enumerate(generators):
            new_p = ps[g] + step
            if new_p < gen["P_min"] or new_p > gen["P_max"]:
                continue
            if gen["poz_low"] <= new_p <= gen["poz_high"]:
                continue
            marginal = abs(fuel_cost(gen, new_p) - fuel_cost(gen, ps[g]))
            if marginal < best_cost:
                best_cost = marginal
                best_g = g
        if best_g is None:
            return None
        ps[best_g] += step

    return ps if sum(ps) == demand else None


def find_best_repaired_solution(pce_solver, bqm, generators, demand, top_n=20):
    """Scan top quantum candidates, repair each, return the cheapest valid one."""
    top_solutions = pce_solver.get_top_solutions(n=top_n, include_decoded=True)
    best = None

    print("\nTop quantum candidates:")
    for i, sol in enumerate(top_solutions, 1):
        if sol.decoded is None:
            continue
        sample = {var: int(val) for var, val in zip(bqm.variables, sol.decoded)}
        ps = [decode_power(g, generators, sample) for g in range(len(generators))]
        cost = sum(fuel_cost(generators[g], ps[g]) for g in range(len(generators)))
        tot = sum(ps)
        valid = (
            tot == demand
            and all(not (generators[g]["poz_low"] <= ps[g] <= generators[g]["poz_high"])
                    for g in range(len(generators)))
        )
        print(f"  {i:2d}. P=[{ps[0]:.0f},{ps[1]:.0f},{ps[2]:.0f}]  "
              f"Tot={tot:.0f}  Cost={cost:.0f}$  "
              f"Prob={sol.prob:.2%}  {'✅' if valid else '❌'}")

        repaired = repair_solution(ps, generators, demand)
        if repaired is not None:
            rep_cost = sum(fuel_cost(generators[g], repaired[g])
                          for g in range(len(generators)))
            if best is None or rep_cost < best[1]:
                best = (repaired, rep_cost, sol.prob)

    return best


result = find_best_repaired_solution(pce_solver, bqm, generators, DEMAND)

if result is not None:
    powers, cost, prob = result
    print(f"\n→ Best repaired: P1={powers[0]:.0f}, P2={powers[1]:.0f}, "
          f"P3={powers[2]:.0f} MW,  Cost={cost:.1f} $  (seed prob={prob:.2%})")
else:
    print("\n⚠️  No valid solution found. Try increasing max_iterations.")

## Step 6 — Compare quantum vs classical

In [ ]:
if result is not None:
    q_powers, q_cost, _ = result
    c_p1, c_p2, c_p3, c_cost = classical_best

    print("=" * 60)
    print(f"  Classical:  P1={c_p1}, P2={c_p2}, P3={c_p3} MW  → ${c_cost:.1f}")
    print(f"  Quantum:    P1={q_powers[0]:.0f}, P2={q_powers[1]:.0f}, "
          f"P3={q_powers[2]:.0f} MW  → ${q_cost:.1f}")
    print("=" * 60)

    if abs(q_cost - c_cost) < 0.1:
        print("\n🎉 PCE-VQE found the global optimum!")
    else:
        print(f"\n⚡ Gap from optimum: {q_cost - c_cost:.1f} $")